# Notebook 10 · A clinical walkthrough of the backend demonstration

**[Product name] — authored virtual patients, evidence-linked feedback, and a simulated learner study.**

This notebook is a presenter walkthrough for a medical professional. It runs the actual offline backend, makes the assumptions visible, and explains Monte Carlo permutation testing (MCPT) with invented learner outcomes. An optional section calls your configured Amazon Bedrock models using your local `.env`.

**Reading key:** **EXECUTED** = a backend operation actually ran. **SIMULATED** = an input or outcome was authored/generated. **NOT EVALUABLE** = evidence or an operational measurement is unavailable. A passing engineering demonstration is not clinical approval or a demonstrated learning benefit.

> **Presenter opening:** “We can show how the system behaves and where its evidence comes from. Today’s learner results are invented test controls. We will show the exact points where qualified reviewers and genuine learners must replace them.”

Clinical route: Patient Studio → encounter and disclosure → feedback/scoring → difficulty/follow-up → a proposed learner study → MCPT → replacement plan. Allow about 12–15 minutes for discussion; fresh live Bedrock execution can take several additional minutes.

## 1 · Start in the right Python environment

In VS Code, select the backend virtual environment as the notebook kernel. Install the backend dependencies and `backend_kelsch/requirements-notebooks.txt` in that environment. For example, from the repository root:

```bash
python -m pip install -r backend_kelsch/requirements-notebooks.txt
python -m pip install nbconvert ipykernel
```

Open this notebook and use **Restart Kernel → Run All**. The default is a complete offline demonstration with no Bedrock requests. For your computer, set `RUN_LIVE_BEDROCK = True` in the next cell and run all cells. Your `.env` can be at the repository root or `backend/.env`; the latter overrides matching file values. Process environment variables take precedence. The live child process explicitly enables the LLM.

A fresh private output directory is chosen on each full run. Restart and run all to create a new receipt rather than overwriting a frozen run. Keep executed notebooks and live traces private; the repository notebook has cleared outputs.

In [ ]:
import os, sys, json, uuid
from pathlib import Path
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "backend_kelsch").is_dir() and (p / "backend/app").is_dir())
sys.path[:0] = [str(root), str(root / "backend")]

RUN_LIVE_BEDROCK = False  # Set True on your computer for billable Bedrock requests.
LIVE_REPORT_PATH = None  # Or load a previous private live_report.json (Path or string).
LIVE_TRACE_PATH = None   # Paired live_trace.json.gz from the SAME run.
PRICING_PATH = None      # Optional private, dated model-pricing JSON; never guessed.
SEED, PERMUTATION_DRAWS, BOOTSTRAP_DRAWS = 2026, 4999, 1999
PRIVATE_BASE = Path(os.environ.get("ROUNDS_NOTEBOOK_PRIVATE_BASE", str(Path.home() / "rounds-private")))
WORKSPACE = PRIVATE_BASE / ("presenter-demo-" + uuid.uuid4().hex[:10])
SIMULATION_DIR, LIVE_DIR = WORKSPACE / "simulation", WORKSPACE / "bedrock"

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML, Markdown
from backend_kelsch.app.study.simulation_demo import run_demo
from backend_kelsch.app.study import presenter as view
plt.rcParams.update({"figure.dpi": 110, "font.size": 11, "axes.labelsize": 11})
get_ipython().run_line_magic("matplotlib", "inline")
display(pd.DataFrame(view.safe_configuration().items(), columns=["Setting", "Safe description"]))
print("Private outputs:", WORKSPACE)
# Never display Settings.model_dump(), .env contents, access keys or raw process logs.

## 2 · Run the backend before showing results

This calls the same simulation runner as the one-command terminal demonstration. It creates versioned inputs, freezes plans and predictions, then executes the API and evaluation code. It does not load a screenshot of an earlier result.

**Clinical meaning:** reproducible source-linked behavior is a prerequisite for review. It is not a substitute for review.

> **Presenter translation:** “We save both the inputs and the system’s predictions before checking answers. That prevents us from quietly changing an answer key after seeing the result.”

In [ ]:
report = run_demo(SIMULATION_DIR, seed=SEED, draws=PERMUTATION_DRAWS, bootstrap_draws=BOOTSTRAP_DRAWS)
inputs = {name: json.loads((SIMULATION_DIR / "inputs" / name).read_text()) for name in ("action_corpus.json", "review_policy.json", "case_preset_decisions.json", "learner_scenarios.json")}
FIGURE_DIR = WORKSPACE / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
def show_figure(fig, name):
    fig.savefig(FIGURE_DIR / (name + ".png"), dpi=150, bbox_inches="tight")
    display(fig)
    plt.close(fig)
display(pd.DataFrame([
    {"Area": "Backend demonstration", "State": report["demo_readiness"], "Interpretation": "Executed offline API and simulation mechanics"},
    {"Area": "Independent human / clinical evidence", "State": "NOT EVALUABLE", "Interpretation": "Synthetic sources cannot close evidence gates"},
    {"Area": "Live Bedrock", "State": "Separate section below", "Interpretation": "Local .env; explicit opt-in or saved paired files"},
    {"Area": "Voice / cloud deployment", "State": "NOT EVALUABLE", "Interpretation": "No microphone, streaming speech or cloud database rehearsal"},
]))
show_figure(view.plot_pipeline(report), "01_backend_flow")

## 3 · Patient Studio and a source-controlled encounter

The rehearsal creates a synthetic draft, rejects premature publication, completes intake, applies existing source-backed laboratory/vital rules, publishes in a **disposable SQLite database**, runs an encounter and debrief, and opens an authored follow-up. It also checks a rejected transition and unavailable text-to-speech fallback.

The script supplies schema authoring statuses only to demonstrate the workflow. **No person clinically approved that generated case.** Existing authored facts and rules supply the content; the model does not invent patient facts.

The clinical review example below is the existing Walter case. Its role, goal and objectives come directly from the versioned case file. These are authored demonstration materials requiring qualified review.

> **Presenter translation:** “The case file is the patient’s source record. The disclosure engine decides what can be revealed. A language model, when enabled, phrases that permitted content and a verifier checks it.”

**Ask the clinician:** What essential content or safety constraint would you need to review before allowing this case into teaching?

In [ ]:
case = json.loads((root / "cases/walter_greene.json").read_text())
display(pd.DataFrame([
    {"Case item": "Title", "Authored value": case["title"]},
    {"Case item": "Student role", "Authored value": case["briefing"]["student_role"]},
    {"Case item": "Encounter goal", "Authored value": case["briefing"]["goal"]},
    {"Case item": "Source", "Authored value": f"cases/walter_greene.json · version {case['version']}"},
]))
display(pd.DataFrame([{ "HTTP rehearsal check": k, "Recorded result": str(v)} for k, v in report["api"].items()]))
domain_scores, scenario_id, domain_tasks = view.domain_table(SIMULATION_DIR)
transcript = domain_tasks[0]["context"]["transcript"]
display(Markdown(f"**Actual offline authored transcript: Walter / {scenario_id}.** This short script is not a real learner encounter."))
display(pd.DataFrame(transcript).rename(columns={"turn":"Turn", "student":"Scripted student", "patient":"Delivered patient reply"}))

## 4 · Optional live Bedrock: inspect attempts, errors, and fallback separately

If `RUN_LIVE_BEDROCK` is true, this section launches a fresh Python process. It reads the local `.env`, enables Bedrock and runs the existing nine authored pipeline scripts plus verifier challenges. It records every classifier/phraser/verifier attempt, including formatting retries. This is an authored-probe evaluation, not a clinical study.

Alternatively, set both saved file paths in cell 1. Paired report/trace mode, version and call denominators are checked. Saved observations remain labeled as saved; they are not newly executed calls.

A model request can fail while a source-linked template still passes mechanical checks. `live_status: EVALUATED` alone does not establish successful model access. The summary below counts successful and failed attempts directly from the trace. A successful reply is still unreviewed. No tokens, dollars, latency or clinical accuracy are synthesized here.

> **Presenter translation:** “There are two separate questions: did the service work, and was the delivered wording clinically sound? This section measures the first and mechanical source checks. Qualified review is still needed for the second.”

In [ ]:
live_report, live_trace = {}, {}
if bool(LIVE_REPORT_PATH) != bool(LIVE_TRACE_PATH):
    raise ValueError("Supply BOTH report and trace from the same saved live run")
if LIVE_REPORT_PATH and LIVE_TRACE_PATH:
    live_report, live_trace, live_summary = view.load_live_files(Path(LIVE_REPORT_PATH).expanduser(), Path(LIVE_TRACE_PATH).expanduser())
    live_summary["Observation source"] = "SAVED RUN — not a fresh request"
elif RUN_LIVE_BEDROCK:
    live_report, live_trace, live_summary = view.run_live_probe(LIVE_DIR, Path(PRICING_PATH).expanduser() if PRICING_PATH else None)
    live_summary["Observation source"] = "FRESH LOCAL PROCESS"
else:
    live_summary = {"Execution": "NOT RUN", "Reason": "Set RUN_LIVE_BEDROCK=True or supply paired saved files", "Clinical interpretation": "Not evaluable from the offline demonstration"}
display(pd.DataFrame([{ "Live check": k, "Recorded result": str(v)} for k, v in live_summary.items()]))
if live_trace and live_report["mode"] == "live" and live_report["usage_and_cost"]["model_calls"]:
    show_figure(view.plot_live(live_trace), "02_live_attempts_and_latency")
    display(pd.DataFrame([{ "Model usage / cost": k, "Recorded value": str(v)} for k, v in live_report["usage_and_cost"].items()]))
    example = live_trace["runs"][0]["turns"][0]
    display(pd.DataFrame([{"Student": example["utterance"], "Delivered reply": example["delivered_reply"], "Cited fact IDs": str(example["cited_fact_ids"]), "Template fallback": example["fallback_used"]}]))

## 5 · Action extraction: what did the student actually say?

The exact span is a quoted portion of student speech, with character start/end offsets. Offsets use an exclusive end. Our generated corpus has 30 invented patient/family groups, two conversations each, and 900 **whole-sentence** spans. A positive action statement is different from an explicit negation.

The split keeps linked patient/family and exact-duplicate material together. A TF-IDF (term frequency–inverse document frequency) representation and logistic classifier train on development groups only. Validation groups are unused; test groups are held out by synthetic identity. Both classifiers share the deliberately easy phrase bank, so perfect scores do not demonstrate performance on new clinical language or arbitrary span boundaries.

**Metric translation:** Precision asks how many predicted actions were supported by the reference. Recall asks how many reference actions were found. F1 balances the two; macro-F1 gives each action label equal weight. Here the reference is an easy authored template bank, so a score of 1.0 does not establish clinical performance.

> **Presenter translation:** “The baseline turns word patterns into numerical features and learns which action labels occur with them. This tests the annotation and comparison machinery. It does not establish that the model understands a clinical encounter.”

**Ask the clinician:** Would the same words express an intention, a completed action, a hypothetical action, or a negation in a real encounter?

In [ ]:
corpus = inputs["action_corpus.json"]
positive = next(a for a in corpus["annotations"] if "prescribe_antibiotic" in a["labels"])
negative = next(a for a in corpus["annotations"] if "not prescribe" in a["span"])
for row in (positive, negative):
    display(HTML(view.span_markup(row["span"], row["start"], row["end"])))
    display(pd.DataFrame([{"Synthetic labels": ", ".join(row["labels"]), "Start": row["start"], "End (exclusive)": row["end"], "Origin": row["label_origin"]}]))
show_figure(view.plot_actions(report), "03_action_spans_and_splits")
critical = report["actions"]["trained_sentence_classifier"]["critical_recall"]
display(pd.DataFrame([{ "Critical action": label, **values} for label, values in critical.items()]))

## 6 · Feedback and the seven-domain rubric

Feedback must point to transcript, documentation or authored source evidence. A citation proves that text was located; it does not prove that a clinical interpretation is correct. The provisional scores below are the existing offline rule-based outputs on the short authored Walter transcript, not the scores used to manufacture learner-study outcomes.

Use **1–4 or N/A**. N/A is not zero. A score of 1 in domain 6 (clinical reasoning) or 7 (professionalism/safety/closure) triggers review. A total is a discussion aid, not a pass/fail grade. Asking about suicidal ideation must not be penalized; an abrupt opening can only raise a communication note.

The synthetic A/B sources and tie breaker exercise review transport. They are correlated outputs constructed from baseline/constructor answers, not three independent clinicians. N/A controls deliberately test disagreement handling.

> **Presenter translation:** “The system can show why it produced a suggestion. A clinician still has to decide whether that suggestion is justified and whether this encounter permits the domain to be rated.”

In [ ]:
display(domain_scores)
claim_tasks = json.loads((SIMULATION_DIR / "review_stage/claim/blinded_tasks.json").read_text())
claim = next(t for t in claim_tasks if t["case"]["case_id"] == "walter_greene")
display(Markdown("**One actual generated feedback claim and its recorded anchors:**"))
display(pd.DataFrame([claim["context"]["candidate_claim"]]).astype(str))
references = json.loads((SIMULATION_DIR / "review_stage/domain/synthetic_a.json").read_text())
na_row = next(r for r in references if r["answer"]["score"] == "N/A")
display(pd.DataFrame([{"Synthetic control": "Unavailable domain", "Reference score": na_row["answer"]["score"], "Reason": na_row["answer"]["na_reason"], "Imputation": "None"}]))
show_figure(view.plot_reviews(report), "04_provisional_review_transport")
semantic = report["reviews"]["batches"]["semantic"]["comparison"]["metrics"]["semantic"]
display(pd.DataFrame([{"Synthetic semantic diagnostic": k, "Count": semantic[k]} for k in ("tp", "fp", "fn", "tn", "denominator")]))
# Constructed reference perturbations also affect these counts; they are not clinical precision estimates.

## 7 · Difficulty and authored progression

The same scripted novice, competent and expert utterances run against estimated straightforward/typical/difficult disclosure thresholds in each case. Clinical facts stay unchanged. The line chart shows discovery counts, not measured learner skill or difficulty. Overlapping curves mean the scripts found the same number of facts.

The separate provisional routing adapter uses rubric thresholds, exact learner-role compatibility, all-N/A abstention and a safety flag. It is not a production recommendation service. The current cases serve distinct professions, so the adapter cannot select a new compatible case; it reports repeat-current-case fallback. Four authored follow-up branches are replayed mechanically.

> **Presenter translation:** “We can make the simulated patient more guarded without changing the patient’s clinical truth. Whether that creates the right level of educational challenge has to be established with actual learners.”

In [ ]:
show_figure(view.plot_difficulty(report), "05_scripted_difficulty")
display(pd.DataFrame([{ "Control": row["control"], **row["recommendation"]} for row in report["difficulty"]["progression"]["controls"]]))
display(pd.DataFrame([{"Authored follow-up replays": report["difficulty"]["authored_followup_replays"], "Uncertain branch records": report["difficulty"]["uncertain_branch_records"], "Clinical content unchanged": report["difficulty"]["clinical_content_unchanged"], "Observed learner difficulty": report["difficulty"]["observed_learner_difficulty"]}]))

## 8 · A proposed learner comparison — with invented outcomes

The question for a future authorized study is whether feedback changes learner performance on a new case. We have **no genuine cohort or validated transfer-case outcomes here**. The generator invents 24 learners, balances the feedback and standard arms, starts scores at 2, adds background changes and an assumed feedback shift, then clips scores to 1–4. Domain 7 is N/A for selected learners.

These scores are generated independently of the backend’s debrief; Bedrock is not grading these invented learners. Baseline/follow-up case IDs are disjoint arithmetic fixture IDs, not approved clinical transfer cases. The protocol, consent and blinded scoring still need the trusted real-evidence workflow.

> **Presenter translation:** “We are putting known numbers through the analysis to see whether it behaves properly. The benefit in this example exists because we wrote it into the generator.”

In [ ]:
scenarios = {s["scenario"]: s for s in inputs["learner_scenarios.json"]["scenarios"]}
beneficial = scenarios["beneficial_effect"]
first_na = next(e["learner_id"] for e in beneficial["encounters"] if e["scores"]["7"] == "N/A")
example_scores = [{"Learner": e["learner_id"], "Arm": e["arm"], "Phase": e["phase"], **{"Domain " + k: v for k,v in e["scores"].items()}} for e in beneficial["encounters"] if e["learner_id"] == first_na]
display(pd.DataFrame(example_scores))
display(view.learner_table(beneficial).head(8))
show_figure(view.plot_learner_changes(beneficial), "06_one_dot_per_learner")

## 9 · Monte Carlo permutation testing: why shuffle learners?

Compute each learner’s mean follow-up-minus-baseline change across domains scored at both phases. Average repeats within domain/phase first. Then calculate the feedback-minus-standard mean change difference.

Under an appropriate randomized parallel-study null model, we ask how often shuffled learner assignments give an absolute difference at least as large. **We shuffle learner outcomes, not individual encounters:** a learner with three encounters still has one analysis weight. Keep arm sizes fixed. The two-sided p-value uses `(extreme draws + 1) / (draws + 1)`, with ties included.

A p-value is not the probability that the treatment works, the probability the null is true, or the size of a benefit. It depends on the outcome and assignment design. A real stratified, clustered or crossover study requires a different approved shuffle plan.

> **Presenter translation:** “If the assignment labels did not explain the outcome, how surprising would this difference be? We repeat the shuffle many times to approximate that answer.”

The histogram recreates the **already-frozen** seed/draw plan after analysis for visualization. It does not tune predictions or outcomes.

In [ ]:
frozen_plan = json.loads((SIMULATION_DIR / "frozen_plan.json").read_text())
result = report["mcpt"]["beneficial_effect"]
assert frozen_plan["scenario_plans"]["beneficial_effect"] == result["plan"]
show_figure(view.plot_permutation(beneficial, result), "07_permutation_null")
display(pd.DataFrame([{ "Frozen analysis item": k, "Value": str(result["plan"][k])} for k in ("outcome", "repeated_encounters", "missingness", "tail", "draws", "bootstrap_draws", "seed")]))

## 10 · Five controls, missingness, and uncertainty

- **No effect:** matched background changes make the arithmetic effect exactly zero and p-value one. This is an exact test control, not a power or false-positive-rate study.
- **Beneficial effect:** a +1 shift is assumed for the feedback arm.
- **Adverse effect:** a −1 shift is assumed; clipping makes the resulting magnitude asymmetric. A non-small p-value does not make that assumed harm disappear.
- **Missingness:** remove selected follow-up phases and one learner’s numeric ratings. Report assigned and scored denominators; do not fill N/A with zero. This is available-case analysis, not complete intention-to-treat inference.
- **Repeated encounters:** add unequal duplicate-score encounters. The effect and p-value match the beneficial scenario because each learner keeps weight one.

**Two distinct uncertainties:** learner-bootstrap intervals describe variation when resampling learner outcomes within arms; Monte Carlo tail intervals describe simulation precision from a finite number of shuffles. Neither validates clinical labels or establishes a real learning effect in invented data.

> **Presenter translation:** “We test that the analysis handles benefit, harm, missing data and repeat visits without changing the unit of evidence. The analysis can be correct while the data still provide no real-world evidence.”

In [ ]:
show_figure(view.plot_mcpt(report), "08_mcpt_controls_and_denominators")
mcpt_rows = []
for name, row in report["mcpt"].items():
    stats, denom = row["statistics"], row["denominators"]
    mcpt_rows.append({"Synthetic control": name, "Assumed shift": row["assumed_feedback_shift"], "Effect": stats["effect"],
        "Bootstrap 95% interval": str(stats["effect_ci95"]), "p-value": stats["p_value"],
        "Monte Carlo tail 95% interval": str({k:stats["monte_carlo_tail_ci95"][k] for k in ("lower","upper")}),
        "Assigned by arm": str(denom["roster_by_arm"]), "Scored by arm": str(denom["scored_by_arm"]),
        "Missing by arm": str(denom["missing_by_arm"]), "Encounters": denom["encounters"], "Seed": stats["seed"]})
display(pd.DataFrame(mcpt_rows))
assert report["mcpt"]["repeated_encounters"]["statistics"] == report["mcpt"]["beneficial_effect"]["statistics"]

## 11 · What can a clinician replace, and exactly where?

The table is the runner’s replacement map. The locations below are relative to the private simulation directory printed at setup. Keep original inputs and receipts; new independent submissions get new versioned configs. Do not change `synthetic` to `human` in old files.

Action/claim/domain pages are offline HTML forms requiring no Python or AWS credentials. The case/preset worksheet is Markdown. Genuine learner data stays behind the existing authorization workflow; this simulation command rejects observed envelopes. A new case, prompt or task wording requires a fresh source-locked review batch.

> **Presenter handoff:** “Please review clinical truth and disclosure timing, whether feedback is supported by its cited evidence, and which rubric domains can actually be observed. We have compartmentalized the assumptions so those judgments can replace them without rewriting the pipeline.”

In [ ]:
replacement = pd.DataFrame(report["replacement_map"])
display(replacement.rename(columns={"component":"Component", "simulation_input":"Relative switch point", "human_adapter":"Genuine replacement route", "remaining":"Evidence still needed"}))
display(pd.DataFrame([{ "Clinical/evidence gate": k, "Current state": v} for k,v in report["evidence_readiness"].items()]))
display(Markdown("**Reviewer handoff paths:**"))
for name in ("action", "claim", "domain"):
    print(name, "→", SIMULATION_DIR / "review_stage" / name / "reviewer_a.html")
print("Clinical/preset worksheet →", SIMULATION_DIR / "review_stage/clinical_and_difficulty_decisions.md")
print("Aggregate HTML →", SIMULATION_DIR / "simulation_report.html")
print("Aggregate JSON →", SIMULATION_DIR / "simulation_report.json")
print("Figures →", FIGURE_DIR)

## 12 · Close the presentation with supported claims

**Supported today:** the actual offline Studio/encounter/debrief/follow-up contracts executed; predictions and plans were frozen; synthetic annotations/references were compared; learner-level statistical mechanics handled all five controls. If enabled, the live section separately records model successes, failures and fallback observations.

**Not established:** clinical approval; independent rating agreement; action extraction on real learner language; full preset calibration; teaching benefit; learner consent/ethics decisions; microphone/streaming voice behavior; PostgreSQL/cloud readiness; voice first-audio latency. Cost is unknown without a sourced complete pricing snapshot.

> **Presenter close:** “This is a reviewable backend demonstration with explicit evidence gaps. The next clinical step is to replace the source judgments, and the next study step is to collect authorized, blinded learner outcomes under the agreed protocol.”

**Discussion prompts:** Which case is worth reviewing first? Which feedback suggestion would you dispute? Which domains should be N/A? What would count as meaningful educational improvement, and what follow-up case would test transfer?

See `backend_kelsch/docs/presenter_notebook.md` for the run instructions, live-output handling and export procedure. Existing historical validation reports retain their original scope.